# Base BANCI-RNPDNO de la presentación

Construye una base auditable con **todas las columnas originales de BANCI** y la fila asociada de RNPDNO, usando el perfil `reporte_diario` y la exportación validada del pipeline. La salida contiene una fila por FUB del bloque de desaparecidos con match en BANCI que usa la presentación.

La relación puede ser de varios registros BANCI a una persona RNPDNO. Para producir una fila por FUB se conserva el registro BANCI principal elegido por el exportador (`id_registro_banci_principal`) y se mantiene `n_registros_banci` para documentar cuántos registros BANCI apuntan a esa persona. El resto de registros BANCI no se pierde en las exportaciones fuente validadas.

## Secciones
1. Configurar entorno y validar insumos.
2. Construir el universo del perfil `reporte_diario`.
3. Declarar rebanadas de la presentación.
4. Inventariar columnas utilizadas por fuente y uso.
5. Crear coincidencias BANCI-RNPDNO por FUB.
6. Verificar estructura, llaves y particiones.
7. Crear reportes de cobertura y validación.
8. Exportar Parquet, Excel y manifiesto.

In [ ]:
from pathlib import Path
import importlib
import sys

import pandas as pd


def encontrar_raiz(inicio: Path) -> Path:
    for candidato in (inicio.resolve(), *inicio.resolve().parents):
        if (candidato / "config.py").is_file() and (candidato / "src" / "pipeline" / "universo.py").is_file():
            return candidato
    raise FileNotFoundError("No se encontró la raíz del proyecto.")


RAIZ = encontrar_raiz(Path.cwd())
sys.path.insert(0, str(RAIZ))

import config
config = importlib.reload(config)

from src.settings import Settings
from src.pipeline import Universo
from src.slices import Slice

PERFIL = "reporte_diario"

INSUMOS_REQUERIDOS = {
    "BANCI original": config.BANCI_FILE,
    "RNPDNO de entrada": config.RNPDNO_MANUAL_FILE_2,
    "plantilla del reporte": config.PLANTILLA_REPORTE_DIARIO,
}
INSUMOS_FALTANTES = [nombre for nombre, ruta in INSUMOS_REQUERIDOS.items() if not Path(ruta).is_file()]
if INSUMOS_FALTANTES:
    detalle = "\n".join(f"- {nombre}: {INSUMOS_REQUERIDOS[nombre]}" for nombre in INSUMOS_FALTANTES)
    raise FileNotFoundError(f"Faltan insumos para reproducir el reporte diario:\n{detalle}")

print(f"Raíz del proyecto: {RAIZ}")
print(f"Perfil de matching: {PERFIL}")
print("Insumos y plantilla encontrados.")

## 2. Cargar el perfil `reporte_diario` y construir el universo

El perfil y el constructor son los mismos que usa el reporte. Si el matching válido está en caché se reutiliza; si cambió el perfil, el código o un insumo, el pipeline actualiza la capa correspondiente.

In [ ]:
cfg = Settings.desde_perfil(PERFIL)
if cfg.perfil != "reporte_diario":
    raise ValueError(f"Perfil inesperado: {cfg.perfil!r}")

print(cfg.resumen())
u = Universo.construir(cfg, verbose=True)

print("\nResumen del universo construido:")
print(u.resumen().to_string(index=False))

## 3. Rebanadas de la presentación y 4. Inventario de columnas

La base final se limita a los FUB de la rebanada `rn-si-ba` del reporte: personas desaparecidas desde octubre de 2024 que sí coinciden con BANCI. También se cuentan las ramas `rn-no-ba` y sus subfiltros de folio para auditar el reparto de las diapositivas 10 y 11. Los nombres de columnas se resuelven desde el universo, no se adivinan.

In [ ]:
REBANADAS_PPT = {
    "coincidencias_diapo_9": Slice("rn-si-ba", "01oct24-max", estatus="des"),
    "sin_match_diapo_10": Slice("rn-no-ba", "01oct24-max", estatus="des"),
    "folio_valido_diapo_10": Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro="cfval"),
    "sin_folio_valido_diapo_11": Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro="sfval"),
    "folio_invalido_capturado": Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro="sfval_sfvac"),
    "folio_vacio": Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro="sfval_cfvac"),
}

conteos_ppt = pd.DataFrame([
    {"componente": nombre, "registros": u.contar(rebanada)}
    for nombre, rebanada in REBANADAS_PPT.items()
])
display(conteos_ppt)


columnas_uso = []
_columnas_vistas = set()


def registrar_columna(fuente, columna, uso, regla="directa"):
    llave = (fuente, columna, uso)
    if llave not in _columnas_vistas:
        columnas_uso.append({
            "fuente": fuente,
            "columna": columna,
            "uso_en_presentacion": uso,
            "regla_o_transformacion": regla,
        })
        _columnas_vistas.add(llave)


for marco, etiqueta in (("rn", "RNPDNO"), ("ba", "BANCI")):
    for dimension in ("fecha", "cve_ent", "entidad", "estatus_loc"):
        registrar_columna(etiqueta, u.columna(marco, dimension), f"dimensión {dimension}")

for token, uso in {
    "cfval": "folio válido",
    "cfvac": "campo de folio vacío",
    "cind": "con indicios de actividad",
    "sisf": "datos insuficientes",
    "dfor": "delito de desaparición forzada",
    "cpar": "delito cometido por particulares",
}.items():
    registrar_columna("RNPDNO", u.columna_bandera("rn", token), uso)

for token, uso in {"dfor": "delito de desaparición forzada", "cpar": "delito cometido por particulares"}.items():
    registrar_columna("BANCI", u.columna_bandera("ba", token), uso)

registrar_columna("RNPDNO", "fub_rnpdno", "llave de persona RNPDNO")
registrar_columna("BANCI", "fub_derivada_rnpdno_banci", "FUB asignado por el matching")
registrar_columna("BANCI", "tipo_match_derivada_rnpdno_banci", "método/estado del match")
registrar_columna("BANCI", "fubs_confirmados_manual_derivada_rnpdno_banci", "FUB adicionales confirmados manualmente")
registrar_columna("BANCI/RNPDNO", "Match con BANCI", "clasificación de salida original", "manual presente → Match completo; estandar/refinado → Match parcial; sin_match → Sin match")

COLUMNAS_USADAS = pd.DataFrame(columnas_uso)
print(f"Columnas canónicas identificadas: {len(COLUMNAS_USADAS)}")
display(COLUMNAS_USADAS)

## 5. Construir coincidencias BANCI ↔ RNPDNO por llave validada

Se usan `construir_llaves_banci` y `construir_llaves_rnpdno` del módulo oficial de exportación, sobre el mismo resultado de matching que construyó `reporte_diario`. BANCI se lee del archivo configurado y RNPDNO del Parquet ETL usado por el universo (`cfg.rnpdno_etl_parquet`). La copia RNPDNO de `config.RNPDNO_MANUAL_FILE_2` está en un corte anterior y la exportación completa la rechaza al detectar FUB ausentes; por eso aquí no se omite esa validación ni se mezclan cortes.

La tabla final toma sólo los FUB de `rn-si-ba` y selecciona el registro indicado por `id_registro_banci_principal`. Para otras filas BANCI asociadas se conserva el conteo `n_registros_banci` y las llaves generadas; no se elige una carpeta por deduplicación arbitraria. Las columnas `estado_final_nuevo` e `idestado_final_nuevo` provienen del mismo universo usado por la presentación. `Match con BANCI` conserva las categorías de la salida original: `sin_match` → `Sin match`; cualquier coincidencia que incluya `manual` → `Match completo`; coincidencias sólo de `estandar` y/o `refinado` → `Match parcial`. Como esta salida sólo incluye coincidencias, no contendrá filas `Sin match`.

In [ ]:
from src.pipeline.exportacion_llaves import construir_llaves_banci, construir_llaves_rnpdno

if u.matching is None:
    raise RuntimeError("El universo no conserva el resultado del matching.")

corte_match = pd.to_datetime(u.meta.get("fecha_corte_rnpdno"), errors="coerce")
corte_match = corte_match.strftime("%Y-%m-%d") if pd.notna(corte_match) else "sin_fecha"
version_match = cfg.clave_matching

# Se reutilizan las mismas llaves y reglas de prioridad que emplea la exportación oficial.
llaves_banci = construir_llaves_banci(u.matching.banci, corte_match, version_match)
llaves_rnpdno = construir_llaves_rnpdno(
    llaves_banci,
    u.matching.fubs_matcheados,
    u.matching.mapa_tipo_match,
    corte_match,
    version_match,
)

# Usar el BANCI original y el Parquet RNPDNO ETL de esta misma configuración.
with pd.ExcelFile(config.BANCI_FILE) as libro_banci:
    hoja_banci = config.BANCI_SHEET or libro_banci.sheet_names[0]
banci_original = pd.read_excel(config.BANCI_FILE, sheet_name=hoja_banci)
rnpdno_etl = pd.read_parquet(cfg.rnpdno_etl_parquet)

if len(banci_original) != len(llaves_banci):
    raise AssertionError(
        f"BANCI original ({len(banci_original):,}) y matching ({len(llaves_banci):,}) "
        "no conservan el mismo número de filas; no se puede asociar por posición."
    )
if "fub_rnpdno" not in rnpdno_etl.columns:
    raise KeyError("El Parquet RNPDNO ETL no tiene la llave fub_rnpdno.")

colisiones_banci = set(banci_original.columns) & set(llaves_banci.columns)
if colisiones_banci:
    raise ValueError(f"BANCI ya contiene nombres reservados para llaves: {sorted(colisiones_banci)}")

banci_con_llaves = pd.concat(
    [banci_original.reset_index(drop=True), llaves_banci.reset_index(drop=True)],
    axis=1,
)

fubs_presentacion = (
    u.slice(REBANADAS_PPT["coincidencias_diapo_9"])["fub_rnpdno"]
    .dropna()
    .astype("string")
    .str.strip()
)
fubs_presentacion = set(fubs_presentacion[fubs_presentacion.ne("")])

rn_keys = llaves_rnpdno.loc[llaves_rnpdno["estatus_match"].eq("con_cruce")].copy()
rn_keys["_fub_join"] = rn_keys.index.astype("string").str.strip()
rn_keys = rn_keys.loc[rn_keys["_fub_join"].isin(fubs_presentacion)].reset_index(drop=True)

rnpdno_etl["_fub_join"] = rnpdno_etl["fub_rnpdno"].astype("string").str.strip()
if rnpdno_etl["_fub_join"].duplicated().any():
    raise AssertionError("El Parquet RNPDNO ETL tiene FUB duplicados; no se puede garantizar una fila por FUB.")
rn_presentacion = rnpdno_etl.merge(
    rn_keys,
    on="_fub_join",
    how="inner",
    validate="one_to_one",
    suffixes=("_rnpdno", "_match"),
)
if set(rn_presentacion["_fub_join"]) != fubs_presentacion:
    faltantes = fubs_presentacion - set(rn_presentacion["_fub_join"])
    raise AssertionError(f"FUB de la presentación ausentes en el RNPDNO ETL: {len(faltantes):,}")

banci_con_llaves["_id_banci_join"] = banci_con_llaves["id_registro_banci"].astype("string").str.strip()
rn_presentacion["_id_banci_join"] = (
    rn_presentacion["id_registro_banci_principal"].astype("string").str.strip()
)
base_coincidencias = rn_presentacion.merge(
    banci_con_llaves,
    on="_id_banci_join",
    how="left",
    validate="many_to_one",
    suffixes=("_rnpdno", "_banci"),
)
base_coincidencias = base_coincidencias.rename(columns={"_fub_join": "fub"})
base_coincidencias = base_coincidencias.drop(columns=["_id_banci_join"])

columnas_estado = ["estado_final_nuevo", "idestado_final_nuevo"]
colisiones_estado = set(columnas_estado) & set(base_coincidencias.columns)
if colisiones_estado:
    raise ValueError(f"La base ya contiene columnas de estado derivado: {sorted(colisiones_estado)}")
estados_universo = u.rn[["fub_rnpdno", *columnas_estado]]
base_coincidencias = base_coincidencias.merge(
    estados_universo, on="fub_rnpdno", how="left", validate="one_to_one",
    indicator="_estado_en_universo",
)
if not base_coincidencias["_estado_en_universo"].eq("both").all():
    raise AssertionError("Hay FUB de la base ausentes en el universo del reporte.")
base_coincidencias = base_coincidencias.drop(columns="_estado_en_universo")

estatus_match = base_coincidencias["estatus_match_rnpdno"].astype("string").str.strip().str.lower()
metodos_match = base_coincidencias["metodos_concurrentes_rnpdno"].astype("string").str.strip().str.lower()
metodo_principal = base_coincidencias["metodo_match_rnpdno"].astype("string").str.strip().str.lower()
metodos_match = metodos_match.where(metodos_match.notna() & metodos_match.ne(""), metodo_principal)
metodos_por_fub = metodos_match.fillna("").str.split(",").map(
    lambda valores: {valor.strip() for valor in valores if valor.strip()}
)
metodos_validos = {"sin_match", "manual", "estandar", "refinado"}
con_match = estatus_match.eq("con_cruce")
metodos_desconocidos = sorted(set().union(*metodos_por_fub) - metodos_validos)
metodos_incompatibles = metodos_por_fub[con_match].map(lambda metodos: not metodos or "sin_match" in metodos).any()
if metodos_desconocidos or metodos_incompatibles:
    raise ValueError(f"Método desconocido, vacío o incompatible con un cruce confirmado: {metodos_desconocidos}")
base_coincidencias["Match con BANCI"] = "Sin match"
con_match_completo = con_match & metodos_por_fub.map(lambda metodos: "manual" in metodos)
con_match_parcial = con_match & metodos_por_fub.map(lambda metodos: bool(metodos & {"estandar", "refinado"}) and "manual" not in metodos)
base_coincidencias.loc[con_match_completo, "Match con BANCI"] = "Match completo"
base_coincidencias.loc[con_match_parcial, "Match con BANCI"] = "Match parcial"

print(f"BANCI: {len(banci_original):,} filas originales | hoja {hoja_banci}")
print(f"RNPDNO ETL: {len(rnpdno_etl):,} filas")
print(f"FUB de la diapositiva 9: {len(fubs_presentacion):,}")
print(f"Coincidencias reunidas: {len(base_coincidencias):,} filas")

## 6. Validaciones estructurales y de consistencia

Se comprueba que las llaves oficiales sean únicas, que todos los FUB de la rebanada presentada tengan una fila en RNPDNO y un registro BANCI principal, y que los cruces/particiones por estado de match y folio cierren con los conteos de `Slice` del pipeline.

In [ ]:
validaciones = []


def registrar_validacion(nombre, pasa, detalle):
    validaciones.append({
        "validacion": nombre,
        "resultado": "PASS" if pasa else "FAIL",
        "detalle": str(detalle),
    })


n_diapo_9 = u.contar(REBANADAS_PPT["coincidencias_diapo_9"])
n_diapo_10_sin_match = u.contar(REBANADAS_PPT["sin_match_diapo_10"])
n_diapo_10_folio_valido = u.contar(REBANADAS_PPT["folio_valido_diapo_10"])
n_diapo_11_sin_folio_valido = u.contar(REBANADAS_PPT["sin_folio_valido_diapo_11"])
n_folio_invalido = u.contar(REBANADAS_PPT["folio_invalido_capturado"])
n_folio_vacio = u.contar(REBANADAS_PPT["folio_vacio"])
n_diapo_9_rn = u.contar(Slice("rn", "01oct24-max", estatus="des"))

registrar_validacion(
    "Rebanada presentada contra FUB exportados",
    len(fubs_presentacion) == n_diapo_9 == len(base_coincidencias),
    f"Slice={n_diapo_9:,}; FUB únicos={len(fubs_presentacion):,}; filas finales={len(base_coincidencias):,}",
)
registrar_validacion(
    "Una fila final por FUB",
    not base_coincidencias["fub"].duplicated().any(),
    f"FUB únicos={base_coincidencias['fub'].nunique():,}; filas={len(base_coincidencias):,}",
)
registrar_validacion(
    "Todos los matches están confirmados",
    base_coincidencias["estatus_match_rnpdno"].eq("con_cruce").all(),
    f"Estados RNPDNO: {base_coincidencias['estatus_match_rnpdno'].value_counts(dropna=False).to_dict()}",
)
registrar_validacion(
    "Clasificación Match con BANCI",
    base_coincidencias["Match con BANCI"].isin(["Match completo", "Match parcial", "Sin match"]).all(),
    f"Categorías: {base_coincidencias['Match con BANCI'].value_counts(dropna=False).to_dict()}",
)
registrar_validacion(
    "Cada FUB tiene registro BANCI principal",
    base_coincidencias["id_registro_banci"].notna().all(),
    f"Sin registro principal: {int(base_coincidencias['id_registro_banci'].isna().sum()):,}",
)
registrar_validacion(
    "Una identidad por fila BANCI",
    not banci_con_llaves["id_registro_banci"].duplicated().any(),
    f"Registros BANCI únicos={banci_con_llaves['id_registro_banci'].nunique():,}",
)
registrar_validacion(
    "Match/no match particionan desaparecidos del sexenio",
    n_diapo_9 + n_diapo_10_sin_match == n_diapo_9_rn,
    f"con match={n_diapo_9:,} + sin match={n_diapo_10_sin_match:,} = {n_diapo_9_rn:,}",
)
registrar_validacion(
    "Folio válido/no válido particionan sin match",
    n_diapo_10_folio_valido + n_diapo_11_sin_folio_valido == n_diapo_10_sin_match,
    f"válido={n_diapo_10_folio_valido:,} + no válido={n_diapo_11_sin_folio_valido:,} = {n_diapo_10_sin_match:,}",
)
registrar_validacion(
    "Folio inválido capturado/vacío particionan sin folio válido",
    n_folio_invalido + n_folio_vacio == n_diapo_11_sin_folio_valido,
    f"capturado={n_folio_invalido:,} + vacío={n_folio_vacio:,} = {n_diapo_11_sin_folio_valido:,}",
)

columnas_duplicadas = base_coincidencias.columns[base_coincidencias.columns.duplicated()].tolist()
registrar_validacion(
    "Nombres de columnas únicos",
    not columnas_duplicadas,
    f"Duplicadas: {columnas_duplicadas}",
)

columnas_banci_salida = set(banci_con_llaves.columns) - {"_id_banci_join"}
columnas_rn_salida = (set(rn_presentacion.columns) - {"_id_banci_join", "_fub_join"}) | {"fub"}
solapadas = columnas_banci_salida & columnas_rn_salida
columnas_banci_esperadas = {
    f"{col}_banci" if col in solapadas else col
    for col in columnas_banci_salida
}
columnas_rn_esperadas = {
    f"{col}_rnpdno" if col in solapadas else col
    for col in columnas_rn_salida
}
columnas_ausentes = (columnas_banci_esperadas | columnas_rn_esperadas) - set(base_coincidencias.columns)
registrar_validacion(
    "Se conservaron todas las columnas fuente",
    not columnas_ausentes,
    f"Columnas BANCI={len(columnas_banci_esperadas):,}; RNPDNO y llave={len(columnas_rn_esperadas):,}; ausentes={sorted(columnas_ausentes)}",
)

VALIDACIONES = pd.DataFrame(validaciones)
display(VALIDACIONES)
if not VALIDACIONES["resultado"].eq("PASS").all():
    raise AssertionError("Una o más validaciones fallaron; no exportar la base.")

## 7. Reporte auditable de columnas, cobertura y validaciones

El inventario enumera cada columna original y las columnas canónicas usadas por la presentación. La cobertura se calcula sin mostrar valores personales; la bitácora deja el resultado y detalle de cada prueba.

In [ ]:
inventario_columnas = []
for fuente, frame in (
    ("BANCI original", banci_original),
    ("BANCI llaves de matching", llaves_banci),
    ("RNPDNO ETL", rnpdno_etl.drop(columns=["_fub_join"], errors="ignore")),
    ("RNPDNO llaves de matching", llaves_rnpdno.reset_index()),
):
    for columna in frame.columns:
        serie = frame[columna]
        inventario_columnas.append({
            "fuente": fuente,
            "columna": columna,
            "tipo": str(serie.dtype),
            "registros": len(frame),
            "no_nulos": int(serie.notna().sum()),
            "cobertura_pct": round(100 * serie.notna().mean(), 2) if len(frame) else 0.0,
        })

INVENTARIO_COLUMNAS = pd.DataFrame(inventario_columnas)

correspondencias = pd.DataFrame([
    {
        "relacion": "FUB de persona",
        "columna_RNPDNO": "fub_rnpdno / id_cruce",
        "columna_BANCI": "id_cruce / fub_en_registro_banci",
        "regla": "FUB confirmado por la cascada de matching del perfil reporte_diario",
    },
    {
        "relacion": "Fila BANCI representativa",
        "columna_RNPDNO": "id_registro_banci_principal",
        "columna_BANCI": "id_registro_banci",
        "regla": "Unión many-to-one; registro principal según prioridad del código oficial",
    },
    {
        "relacion": "Número de filas BANCI asociadas",
        "columna_RNPDNO": "n_registros_banci",
        "columna_BANCI": "id_registro_banci",
        "regla": "Conteo de registros BANCI que apuntan al FUB; no se colapsa por LLAVE",
    },
    {
        "relacion": "Método y evidencia del match",
        "columna_RNPDNO": "metodo_match, submetodo_match, metodos_concurrentes",
        "columna_BANCI": "metodo_match, submetodo_match, metodos_concurrentes, evidencia_match",
        "regla": "Campos producidos por las funciones oficiales de exportación de llaves",
    },
])

metricas_cobertura = pd.DataFrame([
    {"metrica": "Perfil", "valor": cfg.perfil},
    {"metrica": "Clave de matching", "valor": cfg.clave_matching},
    {"metrica": "FUB únicos de la rebanada presentada", "valor": len(fubs_presentacion)},
    {"metrica": "Filas de la base final", "valor": len(base_coincidencias)},
    {"metrica": "BANCI con registro principal localizado", "valor": int(base_coincidencias["id_registro_banci"].notna().sum())},
    {"metrica": "Match completo", "valor": int(base_coincidencias["Match con BANCI"].eq("Match completo").sum())},
    {"metrica": "Match parcial", "valor": int(base_coincidencias["Match con BANCI"].eq("Match parcial").sum())},
    {"metrica": "Sin match", "valor": int(base_coincidencias["Match con BANCI"].eq("Sin match").sum())},
    {"metrica": "Columnas originales de BANCI", "valor": len(banci_original.columns)},
    {"metrica": "Columnas originales del RNPDNO ETL", "valor": len(rnpdno_etl.columns) - int("_fub_join" in rnpdno_etl.columns)},
])

print("Resumen de cobertura:")
display(metricas_cobertura)
print("Relaciones de llaves y procedencia:")
display(correspondencias)
print(f"Inventario de columnas: {len(INVENTARIO_COLUMNAS):,} filas")
display(INVENTARIO_COLUMNAS.head(20))
print("Bitácora de validaciones:")
display(VALIDACIONES)

## 8. Exportar resultados y artefactos para revisión

El Parquet es la salida canónica. El Excel es una copia práctica con pestañas de auditoría. Ambos contienen datos personales y deben resguardarse como información sensible.

In [ ]:
import json
import numbers
from datetime import datetime

if not VALIDACIONES["resultado"].eq("PASS").all():
    raise AssertionError("No se exporta porque hay validaciones fallidas.")

DIRECTORIO_SALIDA = Path(cfg.resultados_dir) / "cruce_banci_rnpdno_presentacion"
DIRECTORIO_SALIDA.mkdir(parents=True, exist_ok=True)
FECHA_EXPORTACION = datetime.now().strftime("%Y%m%d")
NOMBRE_BASE = f"BANCI_RNPDNO_reporte_diario_{FECHA_EXPORTACION}"
RUTA_PARQUET = DIRECTORIO_SALIDA / f"{NOMBRE_BASE}.parquet"
RUTA_EXCEL = DIRECTORIO_SALIDA / f"{NOMBRE_BASE}.xlsx"
RUTA_MANIFIESTO = DIRECTORIO_SALIDA / f"{NOMBRE_BASE}_manifiesto.json"


def valor_a_texto(valor):
    if valor is None or valor is pd.NA:
        return pd.NA
    if isinstance(valor, (list, tuple, set, dict)):
        return json.dumps(valor, ensure_ascii=False, default=str)
    if pd.isna(valor):
        return pd.NA
    return str(valor)


def normalizar_objetos_heterogeneos(frame):
    salida = frame.copy()
    columnas_texto = []
    for columna in salida.select_dtypes(include=["object"]).columns:
        familias = set()
        for valor in salida[columna]:
            if valor is None or valor is pd.NA:
                continue
            if isinstance(valor, (list, tuple, set, dict)):
                familias.add("compuesto")
            elif pd.isna(valor):
                continue
            elif isinstance(valor, str):
                familias.add("texto")
            elif isinstance(valor, numbers.Number):
                familias.add("numero")
            else:
                familias.add(type(valor).__name__)
        if len(familias) > 1 or "compuesto" in familias:
            salida[columna] = salida[columna].map(valor_a_texto).astype("string")
            columnas_texto.append(columna)
    return salida, columnas_texto


base_parquet, columnas_normalizadas = normalizar_objetos_heterogeneos(base_coincidencias)
base_parquet.to_parquet(RUTA_PARQUET, index=False, engine="pyarrow")


def preparar_excel(frame):
    salida = frame.copy()
    for columna in salida.select_dtypes(include=["object"]).columns:
        salida[columna] = salida[columna].map(valor_a_texto)
    return salida


excel_error = None
try:
    with pd.ExcelWriter(RUTA_EXCEL, engine="openpyxl") as writer:
        preparar_excel(base_coincidencias).to_excel(writer, sheet_name="Coincidencias", index=False)
        COLUMNAS_USADAS.to_excel(writer, sheet_name="Columnas_presentacion", index=False)
        correspondencias.to_excel(writer, sheet_name="Llaves_y_relaciones", index=False)
        metricas_cobertura.to_excel(writer, sheet_name="Cobertura", index=False)
        VALIDACIONES.to_excel(writer, sheet_name="Validaciones", index=False)
except Exception as error:
    excel_error = f"{type(error).__name__}: {error}"
    if RUTA_EXCEL.exists():
        RUTA_EXCEL.unlink()

manifiesto = {
    "perfil": cfg.perfil,
    "clave_matching": cfg.clave_matching,
    "clave_universo": cfg.clave_universo,
    "generado": datetime.now().isoformat(timespec="seconds"),
    "fecha_corte_rnpdno": str(u.meta.get("fecha_corte_rnpdno")),
    "fuente_banci": str(config.BANCI_FILE),
    "fuente_rnpdno_etl": str(cfg.rnpdno_etl_parquet),
    "rebanada": REBANADAS_PPT["coincidencias_diapo_9"].nombre,
    "n_fubs": int(base_coincidencias["fub"].nunique()),
    "n_filas": int(len(base_coincidencias)),
    "columnas_objeto_normalizadas_a_texto": columnas_normalizadas,
    "validaciones_pass": int(VALIDACIONES["resultado"].eq("PASS").sum()),
    "validaciones_total": int(len(VALIDACIONES)),
    "parquet": str(RUTA_PARQUET),
    "excel": str(RUTA_EXCEL) if excel_error is None else None,
    "excel_error": excel_error,
}
RUTA_MANIFIESTO.write_text(
    json.dumps(manifiesto, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Parquet: {RUTA_PARQUET}")
if excel_error is None:
    print(f"Excel:   {RUTA_EXCEL}")
else:
    print(f"Excel no se pudo generar: {excel_error}")
print(f"Auditoría: {RUTA_MANIFIESTO}")
print(f"Filas/FUB: {len(base_coincidencias):,} / {base_coincidencias['fub'].nunique():,}")
print(f"Columnas finales: {len(base_coincidencias.columns):,}")
print(f"Columnas heterogéneas serializadas como texto: {columnas_normalizadas}")
print(f"Validaciones: {manifiesto['validaciones_pass']}/{manifiesto['validaciones_total']} PASS")
print("Nombres de columnas:")
print("\n".join(base_coincidencias.columns.astype(str)))

In [ ]:
base_releida = pd.read_parquet(RUTA_PARQUET)
assert len(base_releida) == len(base_coincidencias)
assert base_releida["fub"].nunique() == len(base_releida)
assert list(base_releida.columns) == list(base_parquet.columns)
assert "Match con BANCI" in base_releida.columns
assert base_releida["Match con BANCI"].isin(["Match completo", "Match parcial", "Sin match"]).all()

with pd.ExcelFile(RUTA_EXCEL) as libro_salida:
    hojas_salida = libro_salida.sheet_names
assert {"Coincidencias", "Columnas_presentacion", "Llaves_y_relaciones", "Cobertura", "Validaciones"}.issubset(hojas_salida)

manifiesto_releido = json.loads(RUTA_MANIFIESTO.read_text(encoding="utf-8"))
assert manifiesto_releido["n_fubs"] == len(base_releida)
assert manifiesto_releido["validaciones_pass"] == manifiesto_releido["validaciones_total"] == 11

print("Relectura correcta:")
print(f"- Parquet: {len(base_releida):,} filas, {len(base_releida.columns):,} columnas, FUB único")
print(f"- Excel: {len(hojas_salida)} hojas de datos/auditoría")
print(f"- Manifiesto: {manifiesto_releido['validaciones_pass']}/{manifiesto_releido['validaciones_total']} PASS")